# Inventario de datos de Olist (E-Commerce)

Este notebook arma un mapa general del dataset de Olist antes de entrar en
cualquier mision especifica. No responde ninguna pregunta de negocio todavia:
documenta que tablas hay, que tamano tienen, como se conectan entre si y donde
hay valores nulos. Cada mision (recomendaciones, entregas, sentimiento) parte de
este conocimiento base y hace su propio analisis exploratorio enfocado en su
propia pregunta.

In [ ]:
import pandas as pd
from pathlib import Path

data_raw = Path("..") / "data" / "raw"

In [ ]:
archivos = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

tablas = {nombre: pd.read_csv(data_raw / archivo) for nombre, archivo in archivos.items()}

## Recuento de tablas y su tamano

Cuantas filas y columnas tiene cada tabla. Esto da una primera nocion de la escala
de cada una: por ejemplo, order_items deberia tener mas filas que orders, porque un
pedido puede tener varios items.

In [ ]:
shapes = {nombre: df.shape for nombre, df in tablas.items()}
pd.DataFrame(shapes, index=["filas", "columnas"]).T

## Mapa de relaciones entre tablas

El dataset esta normalizado: cada tabla cubre una entidad, y se conectan entre si
por claves compartidas.

| Tabla A | Clave | Tabla B |
|---|---|---|
| orders | customer_id | customers |
| orders | order_id | order_items |
| order_items | product_id | products |
| order_items | seller_id | sellers |
| orders | order_id | order_payments |
| orders | order_id | order_reviews |
| products | product_category_name | category_translation |
| customers | customer_zip_code_prefix | geolocation (zip_code_prefix) |
| sellers | seller_zip_code_prefix | geolocation (zip_code_prefix) |

orders es la tabla central: casi todo el resto se conecta a ella directa o
indirectamente a traves de order_id o customer_id.

## Perfil de tipos y valores faltantes

Para cada tabla: cuantas columnas hay de cada tipo de dato, y que porcentaje de
valores nulos tiene cada columna. Esto anticipa trabajo de limpieza que cada
mision va a tener que resolver por su cuenta.

In [ ]:
for nombre, df in tablas.items():
    resumen = pd.DataFrame({
        "dtype": df.dtypes,
        "pct_nulos": (df.isna().mean() * 100).round(2),
    })
    print(f"--- {nombre} ---")
    print(resumen)
    print()

## Hoja de ruta de las misiones

A partir de aca, cada mision toma solo las tablas que necesita y hace su propio
analisis exploratorio en un notebook independiente:

- Mision 1 (recomendaciones): order_items, products, category_translation.
- Mision 2 (entregas): orders, order_items, products, sellers, customers, geolocation.
- Mision 3 (sentimiento): order_reviews.